# Gauss-Seidel MDA: converging the models without the optimiser

An MDA (multidisciplinary analysis) converges the coupled models at fixed inputs, with
no optimiser involved. At the end, every quantity that feeds back into an earlier model
equals what the models compute from it. PROCESS does this with its idempotence loop.
`Caller.call_models` repeats the full model sequence in call order until the objective
and the constraints stop changing, at most ten passes. That is a Gauss-Seidel iteration:
the models are swept in order and each uses the newest values. One PROCESS pass is one
sweep.

Here the same iteration is built on the graph of the models. The graph knows which
models feed back into which, so only those are swept and the rest run once. Each
coupled group gets three steps:

1. **Cut** (`FixedPointCut`). Some variables are read by an earlier model in the call
   order than the one that computes them. Each such reader gets a copy, `^hat.x`,
   instead. The group is now a straight sequence with one statement attached: the copy
   must equal the computed value. That statement is a fixed-point problem, bound under
   `^mda`.
2. **Nest** (`Nest`). Some models contain a solve of their own, such as the root find in
   the coil sizing. These are left as they are and converged inside each sweep, as
   inside a PROCESS pass.
3. **Assign a driver**. A driver is the algorithm that solves a problem. Here it is a
   fixed-point iteration (`PicardDriver`): sweep until the copies stop changing, as
   PROCESS's loop does.

Which variables to copy is a choice, and a **scheme** is that choice written as one
graph operation (`cottax.mdao_architectures`). `Jacobi` copies every variable that
crosses between models; every model then reads only last sweep's values, so the group
could run in parallel. `GaussSeidel` copies only what is read before it is computed in a
given order -- binding order by default, which is PROCESS's own call order.
`GaussSeidelMinimal` is Gauss-Seidel in the order that needs the fewest copies, found
exactly; it is what `mda.SCHEME` is, and what the MDF, IDF and SAND notebooks use.

A scheme records a *rule*, not a decision: replayed on a model whose coupling has
changed it derives its cuts again.

This notebook builds the Gauss-Seidel cut in PROCESS's call order, shows the run order
it produces, runs it, and compares the three schemes.

In [1]:
import os
import sys
import time
from pathlib import Path

HERE = Path.cwd()                                   # the notebook's own folder
REPO = next(p for p in (HERE, *HERE.parents) if (p / "functional_process").is_dir())
os.chdir(REPO)                                      # input files are named relative to PROCESS/
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

import jax

jax.config.update("jax_enable_x64", True)          # PROCESS is float64 throughout
try:                                                # whichever cottax is already on the path
    import cottax
except ImportError:                                 # else the checkout beside this repo
    sys.path.insert(0, str(REPO.parent.parent / "jaxgraph" / "src"))
    import cottax
import numpy as np

print("repo  :", REPO)
print("cottax:", Path(cottax.__file__).parent)

repo  : /home/wrutten/projects/functional_PROCESS/PROCESS
cottax: /home/wrutten/projects/jaxgraph/src/cottax


## The graph

The models of the Helias stellarator input file, as the port declares them. Each node
is one model function; it reads and writes PROCESS's own variables (`.physics.rmajor`
is `data.physics.rmajor`). Three groups of models feed back into each other, and one
model contains a solve of its own -- the coil sizing root find, which closes the group
it sits in.

In [2]:
CONFIGURATION = "stellarator_helias"                   # configurations/stellarator_helias.py: the machine, its values, its problem

from functional_process.configurations import load
from functional_process.cottax.architectures.evaluate import without_excluded
from functional_process.cottax.input import native
from functional_process.cottax.input.indat import graph_for
from functional_process.cottax.queries import declared
from functional_process.cottax.visualization.grouping import driver_name, problem_kind

configuration = load(CONFIGURATION)
ref = native.reference_of(configuration)      # the configuration's own values -- PROCESS-free
machine_graph = graph_for(configuration.machine)
raw = without_excluded(machine_graph)

print(f"{len(raw.nodes)} nodes; cyclic components of sizes {[len(c) for c in raw.graph.cycles]}")
print("problems the models declare themselves:")
for p in declared(raw):
    print(f"   {p.spelling:55s} {problem_kind(raw[p])}")

152 nodes; cyclic components of sizes [2, 6, 2, 2, 2]
problems the models declare themselves:
   ^problem.stellarator.coils.intersect                    root-find
   ^problem.physics.profiles.ion_vol_avg_temperature       fixed-point
   ^problem.power.delta_eta_step                           fixed-point


## The recipe

`GaussSeidel()` looks at each coupled group and copies the variables that are read
before they are computed in call order. It is one operation on the graph:
`Plan(raw) + scheme` records the rule, and `scheme.composed(raw)` is what that rule
expands to against *this* graph -- one `FixedPointCut` per group, listed below with
what each cut opened.

In [3]:
from cottax.interfaces import Plan
from cottax.mdao_architectures import GaussSeidel, GaussSeidelMinimal, Jacobi, coupled

from functional_process.architecture_examples.notebook_tools import print_recipe
from functional_process.cottax.architectures.mda import cut_graph

SCHEMES = {
    "jacobi": Jacobi(),                          # every coupling of every cycle
    "gauss_seidel": GaussSeidel(),               # the back edges of binding order (PROCESS's call order)
    "gauss_seidel_minimal": GaussSeidelMinimal(),  # the order that cuts the fewest
}
scheme = SCHEMES["gauss_seidel"]

print(f"{len(coupled(raw))} cycle(s) for the scheme to open:")
for closure in scheme.composed(raw):
    print(f"   {closure.problem.spelling}: {len(closure.cuts)} variable(s) cut")
    for c in closure.cuts:
        print(f"      cut {c.var.spelling:45s} read by {[n.spelling for n in c.readers]}")

plan = Plan(raw) + scheme
print("\nthe plan:")
print_recipe(plan)
print("\nsame graph as mda.cut_graph(raw, scheme):", plan.graph == cut_graph(raw, scheme))
print("problems the cut minted:", [p.spelling for p in declared(plan.graph) if p not in set(declared(raw))])

component of 2 nodes: 0 variable(s) cut over 0 read(s) -> ^problem.physics.profiles.ion_vol_avg_temperature
component of 6 nodes: 7 variable(s) cut over 7 read(s) -> ^problem.physics.fusion_power_totals_mw.mda
      cut .physics.fusden_plasma                        read by ['.physics.fusion_totals_no_beam']
      cut .physics.fusden_plasma_alpha                  read by ['.physics.fusion_totals_no_beam']
      cut .physics.dt_power_density_plasma              read by ['.physics.fusion_power_totals_mw']
      cut .physics.dhe3_power_density                   read by ['.physics.fusion_power_totals_mw']
      cut .physics.dd_power_density                     read by ['.physics.fusion_power_totals_mw']
      cut .physics.nd_plasma_fuel_ions_vol_avg          read by ['.physics.fusion_rates']
      cut .physics.nd_plasma_ions_total_vol_avg         read by ['.physics.profiles.parameterisation.parabolic_on_axis_densities']
component of 2 nodes: 0 variable(s) cut over 0 read(s) -> nothing to do

Two of the three groups needed copies: seven variables in the fusion-power group, one
in the first-wall group. The third is the coil sizing loop, whose only coupling is the
root find's own unknown read by the model it drives -- a copy of it would be a second
unknown equal to the first, so the scheme leaves that group alone and the root find
closes it.

### Assign the drivers

`default_drivers` picks a driver by what makes an answer right: fixed-point iteration
for a consistency statement, Newton for a root find. `assign_drivers` attaches them in
the graph. The iteration is asked to report how many sweeps it took, so the count comes
out of the run like any other value. The graph's components below are its blocks in run
order, one block per iterated group; `RunnableGraph` is the proof each can be answered
*and* run here. A schedule is that order made runnable.

In [4]:
from cottax.interfaces import RunnableGraph, Schedule
from cottax.visualization import problems_at

from functional_process.cottax.architectures.drivers import PicardDriver
from functional_process.cottax.architectures.mda import assign_drivers, default_drivers

drivers = default_drivers(plan.graph)
for problem, driver in drivers.items():
    if isinstance(driver, PicardDriver):
        drivers[problem] = PicardDriver(report_steps=True)
runnable = assign_drivers(plan.graph, drivers)     # one `Assign` per problem, plus `Rename` of a start the graph computes
proof = RunnableGraph(runnable)
schedule = Schedule(proof)

for problem, block in zip(problems_at(runnable), runnable.graph.components, strict=True):
    if problem is not None:
        print(f"{problem.spelling:55s} {len(block):3d} nodes  {driver_name(runnable[problem])}")
print(f"{len(runnable.graph.components)} blocks, {len(schedule.steps)} schedule steps, {len(schedule.inputs)} inputs")

^problem.physics.profiles.ion_vol_avg_temperature         2 nodes  PicardDriver
^problem.physics.fusion_power_totals_mw.mda               7 nodes  PicardDriver
^problem.stellarator.coils.intersect                      2 nodes  SeededNewtonDriver
^problem.stellarator.fw_area.mda                          3 nodes  PicardDriver
^problem.power.delta_eta_step                             2 nodes  PicardDriver
143 blocks, 143 schedule steps, 310 inputs


## The process

The design structure matrix (DSM) in run order. Every model is a row. A mark below the
diagonal is a feed-forward, a mark above it a feedback. Iterated groups are boxed and
labelled with their driver. Everything else runs once, in an order derived from what
each model reads and writes. The DSM is written as an interactive page next to this notebook. In the page, hover a
cell for the variables it carries and click a box to fold it.

In [5]:
from functional_process.cottax.visualization.grouping import (
    render_grouped_dsm_html,
    structure_order,
)
from functional_process.cottax.visualization.render_xdsm import SPELLING

dsm = render_grouped_dsm_html(
    proof, order=structure_order(proof),
    title="stellarator_helias -- Gauss-Seidel MDA (call order), run order",
    file_name="dsm_mda_gauss_seidel", outdir=str(HERE), write=True, formatter=SPELLING,
)
print("written:", dsm.path)

Using adapted ragraph from debug branch
written: /home/wrutten/projects/functional_PROCESS/PROCESS/functional_process/architecture_examples/mda_gauss_seidel/dsm_mda_gauss_seidel.html


## Run it

The run starts from the state after one pass in call order, where PROCESS starts too,
and runs as one compiled program. The sweep counts come out of the result: a driver's
report is an ordinary variable of the graph, minted under `^steps`, so
`mdf.report_place` asks the node where it landed. The converged values are compared one
by one with those the default scheme (`GaussSeidelMinimal`) reaches.

In [6]:
from cottax.interfaces import PathMap, Steps

from functional_process.cottax.architectures import mdf
from functional_process.cottax.architectures.evaluate import (
    cold_state,
    jit_schedule,
    mda_env,
    seed_env,
)


def sweeps_of(graph, out):
    """How many sweeps each Picard took, out of a run's env: its `^steps.<u>` report."""
    found = {}
    for p in declared(graph):
        place = mdf.report_place(graph[p], Steps)
        if place is not None and place in out:
            found[p.spelling] = int(out[place])
    return found

env = PathMap(seed_env(ref.data, schedule, runnable, cold_state(ref.data, machine_graph)))
run = jit_schedule(schedule)

began = time.perf_counter()
out = dict(run(env))
print(f"first run {time.perf_counter() - began:.1f} s (compiles)")
began = time.perf_counter()
out = dict(run(env))
print(f"warm run  {(time.perf_counter() - began) * 1000:.1f} ms\n")

steps = sweeps_of(runnable, out)
for block, n in steps.items():
    print(f"Picard steps {n:3d}   {block}")

_, base = mda_env(ref, graph=machine_graph)         # the default scheme's fixed point
worst, compared = 0.0, 0
for var, value in out.items():
    if var.spelling.startswith("^") or var not in base:
        continue
    a, b = np.asarray(value, float), np.asarray(base[var], float)
    if a.shape != b.shape or not a.size or (np.all(a == 0) and np.all(b == 0)):
        continue
    compared += 1
    worst = max(worst, float(np.max(np.abs(a - b) / np.maximum(np.abs(b), 1e-300))))
print(f"\n{compared} variables compared with the minimal scheme's MDA; worst relative difference {worst:.1e}")

first run 2.1 s (compiles)
warm run  6.8 ms

Picard steps   2   .physics.profiles.ion_vol_avg_temperature
Picard steps   3   .physics.fusion_power_totals_mw.mda
Picard steps   5   .stellarator.fw_area.mda
Picard steps   2   .power.delta_eta_step



705 variables compared with the hand cut's MDA; worst relative difference 2.4e-07


## The three schemes side by side

Same models, same start, same iteration; only the copied variables differ. `steps` is
the number of sweeps per group. `depth` is how many models of the group run one after
another within a sweep: one for Jacobi, the sweep length for Gauss-Seidel. So
`steps x depth` counts sequential model evaluations.

In [7]:
import networkx as nx
from cottax.interfaces import body_of


def measure(scheme):
    graph = cut_graph(raw, scheme)
    drivers = default_drivers(graph)
    for problem, driver in drivers.items():
        if isinstance(driver, PicardDriver):
            drivers[problem] = PicardDriver(report_steps=True)
    driven = assign_drivers(graph, drivers)
    sched = Schedule(RunnableGraph(driven))
    values = PathMap(seed_env(ref.data, sched, driven, cold_state(ref.data, machine_graph)))
    out = dict(jit_schedule(sched)(values))
    counts = sweeps_of(driven, out)
    depths = {}
    for component, problem in zip(driven.graph.components, problems_at(driven), strict=True):
        if problem is not None and len(component) > 1:
            body = body_of(driven.subgraph(component))
            depths[problem.spelling] = nx.dag_longest_path_length(body.graph._nx_dependencies) + 1
    return {"steps": counts, "total": sum(counts.values()),
            "sequential": sum(counts.get(b, 0) * d for b, d in depths.items()), "depths": depths}

table = {name: measure(s) for name, s in SCHEMES.items()}

print(f"{'scheme':22s} {'blocks':>6s} {'steps':>6s} {'steps x depth':>14s}   per block")
for name, m in table.items():
    print(f"{name:22s} {len(m['steps']):6d} {m['total']:6d} {m['sequential']:14d}   {list(m['steps'].values())}")

RESULT = {"steps": steps, "worst_relative_difference": worst, "compared": compared,
          "recipes": {k: {"total": v["total"], "sequential": v["sequential"]} for k, v in table.items()}}
RESULT

cut                    blocks  steps  steps x depth   per block
hand (mda.CUTS)             4     12             32   {'.physics.profiles.ion_vol_avg_temperature': 2, '.physics.proton_rate_density.cycle': 3, '.fwbs.f_ster_div_single': 5, '.power.delta_eta_step': 2}
jacobi                      4     17             17   {'.physics.profiles.ion_vol_avg_temperature': 2, '.physics.fusion_power_totals_mw.mda': 5, '.stellarator.fw_area.mda': 8, '.power.delta_eta_step': 2}
gauss_seidel                4     12             26   {'.physics.profiles.ion_vol_avg_temperature': 2, '.physics.fusion_power_totals_mw.mda': 3, '.stellarator.fw_area.mda': 5, '.power.delta_eta_step': 2}
gauss_seidel_minimal        4     11             26   {'.physics.profiles.ion_vol_avg_temperature': 2, '.physics.fusion_power_totals_mw.mda': 2, '.stellarator.fw_area.mda': 5, '.power.delta_eta_step': 2}


{'steps': {'.physics.profiles.ion_vol_avg_temperature': 2,
  '.physics.fusion_power_totals_mw.mda': 3,
  '.stellarator.fw_area.mda': 5,
  '.power.delta_eta_step': 2},
 'worst_relative_difference': 2.3689166620568197e-07,
 'compared': 705,
 'recipes': {'hand (mda.CUTS)': {'total': 12, 'sequential': 32},
  'jacobi': {'total': 17, 'sequential': 17},
  'gauss_seidel': {'total': 12, 'sequential': 26},
  'gauss_seidel_minimal': {'total': 11, 'sequential': 26}}}

Every scheme reaches the same converged state. They differ in how many sweeps it takes
and how long a sweep is. The graph decides where copies are needed, the scheme decides
which variables to copy, and the table shows what that decision costs.